# Experiment: Naive Paraphrasers and False Positives
The goal of this experiment is to find out whether Naive Paraphrasers risk False Positives because they (hypothesis) know too much about the original text.
Hence, we hypothesize if the imposter model uses only Naive Paraphrasers (i.e. T5, Ollama, etc.) to generate imposters, it will likely lead to a high number of False Positives.
We therefore manually comment out the Non-Naive LLM-based imposter generators in the `LLMImposterGenerator` class (`ImposterGenerator.py`) and use only the Naive Paraphrasers.

## Dataset
We construct a dataset of original texts with candidates from a different author (i.e. their LLM generated paraphrase). The dataset should include:
- Original text
- Paraphrased text
- Label indicating which paraphraser generated the paraphrase

The original texts should be from a variety of sources to ensure diversity in genres.

Since we ensured that the Imposter Generators are only using Naive Paraphrasers, we can generate candidates via Non-Naive LLM-based Paraphrasers (e.g. TopicParaphraser, BulletPointParaphraser, etc.) to see if they lead to False Positives.

In [ ]:
import os
from pathlib import Path
from genai_detection.config import CONFIG
from datasets import load_from_disk

# The construction of this cross-genre dataset is in file `genai_detection/dataset_util.py`.
dataset = load_from_disk(Path(os.getcwd()).resolve().parent / CONFIG.CROSS_GENRE)[
    "train"
].to_pandas()
print(f"Loaded Cross-genre dataset with {len(dataset)} examples.")
display(dataset)

In [ ]:
# import os
# from pathlib import Path
# import pandas as pd
# from genai_detection.util import preprocess_text as _preprocess_text


# category2directory = {  # value: directory name and one file name in the datasets folder
#     "News": [
#         "custom_texts",
#         "cnn_040725",
#     ],  # Dalai Lama; "cnn_230625"    # USA attacks Iran
#     "Blog": ["blog_corpus", 27],
#     "Gutenberg": ["gutenberg", "Othello_the_Moor_of_Venice_William_Shakespeare"],
#     "Student Essay": [
#         "student_essays/Intro2006",
#         "Ass1/2006_AB4847",
#     ],  # stream of consciousness essay: 18 years old girl who writes about her bipolar ex-boyfriend of 8 months and her new boyfriend who loves music like her but is not catholic; she wants to live in the present
# }

# dataset = pd.DataFrame(
#     columns=[
#         "category",
#         "original_text",
#         "original_file_name",
#         "text_length",
#         # "paraphrased_text",
#         # "paraphraser",
#     ]
# )


# def load_text(data_category, path, file_name):
#     if data_category == "Blog":
#         df = pd.read_csv(path / "blogtext.csv")
#         return df.loc[file_name, "text"]
#     return open(path / f"{file_name}.txt", encoding="utf-8").read()


# for data_category in category2directory.keys():
#     path2datasets = (
#         Path(os.getcwd()).resolve().parent
#         / "data"
#         / "datasets"
#         / category2directory[data_category][0]
#     )
#     dir_name, file_name = category2directory[data_category]
#     text = " ".join(
#         _preprocess_text(load_text(data_category, path2datasets, file_name)).split()[
#             :1221
#         ]
#     )
#     dataset = pd.concat(
#         [
#             dataset,
#             pd.DataFrame(
#                 [
#                     {
#                         "category": data_category,
#                         "original_text": text,
#                         "original_file_name": file_name,
#                         "text_length": len(text.split()),
#                     }
#                 ]
#             ),
#         ]
#     )
# dataset.reset_index(drop=True, inplace=True)
# display(dataset)

,category,original_text,original_file_name,text_length
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582
1,Blog,this may be a long blog...got a lot of thought...,27,933
2,Gutenberg,venice. a street. enter roderigo and iago. tus...,Othello_the_Moor_of_Venice_William_Shakespeare,1221
3,Student Essay,"the thoughts, feelings, and perceptions that o...",Ass1/2006_AB4847,1221


In [ ]:
# import sys

# from tqdm import tqdm
# from genai_detection.config import CONFIG
# from genai_detection.paraphrasing.paraphraser import (
#     BulletPointParaphraser,
#     OllamaParaphraser,
#     T5ChatGPTParaphraser,
#     T5GooglePAWSParaphraser,
#     TaskParaphraser,
#     TitleParaphraser,
#     TopicParaphraser,
#     TranslationParaphraser,
# )

# OLLAMA_VERSION = "zephyr:7b"  # "mistral:7b"  # "default:latest"
# # PROMPTS = [
# #     "Paraphrase the text above and output only the paraphrased version.",
# #     "For the text above: First, extract bullet points capturing the main ideas, then create a text based on these bullet points. Only output the final text (i.e. do not output the bullet points or any additional chain of thoughts).",
# #     "For the text above: Paraphrase the sentence by first identifying the main subject, verb, and object. Then find synonyms for each and construct a new sentence. Only output the final paraphrased sentence.",
# #     "For the text above: Paraphrase the sentence using the same tone as the original with approximately the same number of words.",
# #     "For the text above: Paraphrase this sentence. Do not change the meaning, but use different words and structure. Output only the paraphrased sentence.",
# # ]
# temperature = CONFIG.TEMPERATURE


# def create_paraphrasers(model):
#     return {
#         # "T5_ChatGPT": T5ChatGPTParaphraser(),
#         # "T5_Google_PAWS": T5GooglePAWSParaphraser(),
#         "Ollama": model,
#         "BulletPoint": BulletPointParaphraser(model, model),
#         # "Task": TaskParaphraser(model, model),
#         # "Topic": TopicParaphraser(model, model),
#         # "Title": TitleParaphraser(model, model),
#         "Translation": TranslationParaphraser(model, model),
#     }


# new_rows = []
# model = OllamaParaphraser(model_id=OLLAMA_VERSION)
# paraphrasers = create_paraphrasers(model)
# for i in tqdm(dataset.index, desc="Processing dataset"):
#     text = dataset.loc[i, "original_text"]
#     # print(
#     #     f"Processing text from {dataset.loc[i, 'category']} category, file: {dataset.loc[i, 'original_file_name']}, length: {dataset.loc[i, 'text_length']} words"
#     # )
#     # for prompt in PROMPTS:

#     paraphrase_config = {
#         "text": text,
#         "n_responses": 1,
#         "prompt": "",
#         "temperature": temperature,
#     }
#     for paraphraser_name, paraphraser in paraphrasers.items():
#         print(f"Paraphrasing with {paraphraser_name}...")
#         try:
#             paraphrase = _preprocess_text(
#                 paraphraser.paraphrase(**paraphrase_config)[0]
#             )
#         except Exception as e:
#             print(f"[ERROR] Failed to paraphrase with {paraphraser_name}: {e}")
#             continue

#         # Save a copy of the original row + new info
#         new_row = dataset.loc[i].copy()
#         # new_row["prompt"] = None
#         new_row["candidate_paraphraser"] = paraphraser_name
#         new_row["paraphrased_text"] = paraphrase
#         new_rows.append(new_row)

# # Create a new DataFrame from the new rows
# expanded_dataset = pd.DataFrame(new_rows)
# expanded_dataset.reset_index(drop=True, inplace=True)
# # (Optional) Save or display
# display(expanded_dataset.head())
# # expanded_dataset.to_csv("paraphrased_dataset.csv", index=False)

Processing dataset:   0%|          | 0/4 [00:00<?, ?it/s]

Paraphrasing with Ollama...
Paraphrasing with BulletPoint...
Paraphrasing with Translation...


Processing dataset:  25%|██▌       | 1/4 [03:13<09:41, 193.85s/it]

Paraphrasing with Ollama...
Paraphrasing with BulletPoint...
Paraphrasing with Translation...


Processing dataset:  50%|█████     | 2/4 [05:02<04:47, 143.96s/it]

Paraphrasing with Ollama...
Paraphrasing with BulletPoint...
Paraphrasing with Translation...


Processing dataset:  75%|███████▌  | 3/4 [08:19<02:48, 168.03s/it]

Paraphrasing with Ollama...
Paraphrasing with BulletPoint...
Paraphrasing with Translation...


Processing dataset: 100%|██████████| 4/4 [09:39<00:00, 144.97s/it]


,category,original_text,original_file_name,text_length,candidate_paraphraser,paraphrased_text
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Ollama,the potential successor to the dalai lama has ...
1,News,"for much of the past century, the dalai lama h...",cnn_040725,582,BulletPoint,"in a striking development, the spiritual leade..."
2,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Translation,"'pendant plus d'un siecle, le dalai-lama a ete..."
3,Blog,this may be a long blog...got a lot of thought...,27,933,Ollama,the author's reflections on connections and de...
4,Blog,this may be a long blog...got a lot of thought...,27,933,BulletPoint,it's always fascinating for me to encounter co...


## AV Models
Multiple AV models should be used to evaluate the paraphrased texts.

In [ ]:
import os
from pathlib import Path
import numpy as np
from tqdm import tqdm
from genai_detection.detectors.impostor import ImpostorDetector
from genai_detection.detectors.unmasking import UnmaskingDetector
from genai_detection.detectors.ppmd import PPMdDetector

naive_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="naive_llm",  # Use only naive paraphrasers for imposter generation
)
non_naive_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="non_naive_llm",  # Use only non-naive paraphrasers for imposter generation
)
generalized_imposter_detector = ImpostorDetector(
    path2imp=Path(os.getcwd()).resolve().parent / CONFIG.PATH2BLOG,
    imposter_technique="llm",  # Use all paraphrasers for imposter generation
)
# unmasking_detector = UnmaskingDetector()
# ppmd_detector = PPMdDetector()


# for detector in [imposter_detector, unmasking_detector, ppmd_detector]:
for detector_name, detector in zip(
    ["Naive", "Non-Naive", "Generalized"],
    [
        naive_imposter_detector,
        non_naive_imposter_detector,
        generalized_imposter_detector,
    ],
):
    dataset[f"{detector_name}_score"] = np.nan
    for i in tqdm(
        dataset.index,
        desc=f"Processing {detector_name} scores",
    ):
        original_text = dataset.loc[i, "disputed_text"]
        paraphrased_text = dataset.loc[i, "candidte_text"]
        score = detector.get_score([original_text, paraphrased_text], normalize=False)
        dataset.loc[i, f"{detector_name}_score"] = np.round(score, 2)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Processing ImpostorDetector scores: 100%|██████████| 12/12 [20:37<00:00, 103.12s/it]


In [ ]:
save_path = (
    Path(os.getcwd()).resolve().parent / "results" / "paraphrasing" / "experiments"
)
if not save_path.exists():
    save_path.mkdir(parents=True, exist_ok=True)
display(dataset.head())
dataset.to_csv(save_path / "exp_imp_gen_av_scores_cross_genre_dataset.csv", index=False)

,category,original_text,original_file_name,text_length,candidate_paraphraser,paraphrased_text,ImpostorDetector_score
0,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Ollama,the potential successor to the dalai lama has ...,100.0
1,News,"for much of the past century, the dalai lama h...",cnn_040725,582,BulletPoint,"in a striking development, the spiritual leade...",100.0
2,News,"for much of the past century, the dalai lama h...",cnn_040725,582,Translation,"'pendant plus d'un siecle, le dalai-lama a ete...",100.0
3,Blog,this may be a long blog...got a lot of thought...,27,933,Ollama,the author's reflections on connections and de...,100.0
4,Blog,this may be a long blog...got a lot of thought...,27,933,BulletPoint,it's always fascinating for me to encounter co...,100.0


In [9]:
# calculate rmse per paraphraser and AV model
rmse_per_paraphraser = {}
for paraphraser in expanded_dataset["candidate_paraphraser"].unique():
    rmse_per_paraphraser[paraphraser] = {}
    for detector in [imposter_detector]:
        scores = expanded_dataset[
            expanded_dataset["candidate_paraphraser"] == paraphraser
        ][f"{detector.__class__.__name__}_score"]
        rmse = np.sqrt(np.mean((scores - 0) ** 2))
        rmse_per_paraphraser[paraphraser][detector.__class__.__name__] = rmse
# to dataframe
rmse_df = pd.DataFrame(rmse_per_paraphraser).T
rmse_df.reset_index(inplace=True)
rmse_df.rename(columns={"index": "candidate_paraphraser"}, inplace=True)
# rmse_df.to_csv(save_path / "naive_paraphrase_rmse_per_paraphraser.csv", index=False)
print("RMSE per candidate_paraphraser:")
display(rmse_df)

RMSE per candidate_paraphraser:


,candidate_paraphraser,ImpostorDetector
0,Ollama,100.000000
1,BulletPoint,100.000000
2,Translation,95.924775
